In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("silver.stock_time_series")

In [0]:
# --- Window for rolling calculations ---
# Window functions are core PySpark — important to understand
window_7d = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-6, 0)
window_30d = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-29, 0)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
df_gold = df \
    .withColumn("ma_7d", F.avg("Close").over(window_7d)) \
    .withColumn("ma_30d", F.avg("Close").over(window_30d)) \
    .withColumn("daily_return",
        (F.col("close") - F.lag("Close", 1).over(
            Window.partitionBy("Ticker").orderBy("Date"))) 
        / F.lag("Close", 1).over(
            Window.partitionBy("Ticker").orderBy("Date"))) \
    .withColumn("volatility_7d", F.stddev("Close").over(window_7d)) \
    .withColumn("gold_created_at", F.current_timestamp())

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.stock_time_series")

display(df_gold.orderBy("Date", ascending=False).limit(10))